# 12 — Ensemble Learning & Model Stacking

**Objective:** Construct Voting and Stacking ensembles combining top tree-based models and evaluate whether ensembling outperforms the best single model.


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.ensemble import VotingRegressor
from sklearn.multioutput import MultiOutputRegressor

def find_root():
    p = Path.cwd()
    for parent in [p] + list(p.parents):
        if (parent / 'data' / 'raw' / 'gt_2011.csv').exists():
            return parent
    return p

ROOT = find_root()
sys.path.insert(0, str(ROOT))

from src.models.model_factory import create_model
from src.data.loader import FEATURES_A, TARGETS_A
from src.data.splitting import temporal_split
from src.evaluation.metrics import regression_metrics


## 1. Building Ensemble Regressor


In [2]:
df = pd.read_parquet(ROOT / "data" / "processed" / "cleaned_master.parquet")
dev, cal, test = temporal_split(df)

m1 = create_model("HistGradientBoosting", quick=True)
m2 = create_model("RandomForest", quick=True)

m1.fit(dev[FEATURES_A], dev[TARGETS_A].values)
m2.fit(dev[FEATURES_A], dev[TARGETS_A].values)

p1 = m1.predict(cal[FEATURES_A])
p2 = m2.predict(cal[FEATURES_A])
p_ensemble = 0.6 * p1 + 0.4 * p2

m1_metrics = regression_metrics(cal[TARGETS_A].values, p1, TARGETS_A)
ens_metrics = regression_metrics(cal[TARGETS_A].values, p_ensemble, TARGETS_A)

records = []
for m in m1_metrics:
    records.append({"model": "HistGradientBoosting_Single", **m})
for m in ens_metrics:
    records.append({"model": "HGB_RF_Weighted_Ensemble", **m})

df_ens = pd.DataFrame(records)
print(df_ens[["model", "target", "RMSE", "MAE", "R2"]].to_string(index=False))

out_ens = ROOT / "experiments" / "ensemble_comparison.csv"
df_ens.to_csv(out_ens, index=False)
print(f"Saved ensemble comparison to {out_ens.relative_to(ROOT)}")
print("Ensemble learning comparison COMPLETE.")


                      model target    RMSE    MAE      R2
HistGradientBoosting_Single     CO  1.2620 0.7283  0.6045
HistGradientBoosting_Single    NOX 10.6522 9.3462 -0.1413
   HGB_RF_Weighted_Ensemble     CO  1.2803 0.7157  0.5930
   HGB_RF_Weighted_Ensemble    NOX 10.4891 9.1893 -0.1066
Saved ensemble comparison to experiments\ensemble_comparison.csv
Ensemble learning comparison COMPLETE.
